# SemiSupCon: Semi-Supervised Contrastive Learning of Musical Representations

**Paper:** Guinot, Quinton & Fazekas (ISMIR 2024)  
**Réplica a escala completa** en Google Colab Pro.

| Componente | Configuración |
|---|---|
| $\mathcal{U}$ (no supervisado) | FMA Medium — 25,000 clips × 30s |
| $\mathcal{S}$ (supervisado) | MagnaTagATune — Top 50 tags, split 12:1:3 |
| Encoder | SampleCNN ($d_E=512$) |
| Projection | MLP 2 capas ($d_g=128$) + L2 norm |
| Training | 200K steps, batch 96, $\tau=0.1$, Adam $lr=10^{-4}$ |
| Evaluación | MTAT tagging (AUROC/AP) + GTZAN género (Accuracy) |

---

## 1. Setup e Imports

In [3]:
import os, sys, random, zipfile, subprocess, urllib.request, warnings, json, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler, ConcatDataset, TensorDataset
import torchaudio
import librosa
from scipy.signal import butter, lfilter
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.manifold import TSNE
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Reproducibility ──
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/my_paper_data')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = DRIVE_BASE / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
CKPT_DIR = RESULTS_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

# ── Device ──
assert torch.cuda.is_available(), 'GPU requerida — activa GPU en Runtime > Change runtime type'
DEVICE = torch.device('cuda')
print(f'PyTorch {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

# ═══════════════════════════════════════════════════
# Hyperparameters — paper Section 4.2, 4.3
# ═══════════════════════════════════════════════════
SR = 22050                       # Sample rate
DURATION = 2.7                   # Segment duration (seconds)
N_SAMPLES = int(SR * DURATION)   # 59535 samples
D_ENCODER = 512                  # Encoder embedding dim
D_PROJ = 128                     # Projection head output dim
TAU = 0.1                        # Contrastive temperature τ
LR = 1e-4                        # Learning rate
BATCH_SIZE = 96                  # Batch size (paper)
MAX_STEPS = 200_000              # Training steps (paper)
BS_RATIO = 0.5                   # Supervised proportion per batch
NUM_WORKERS = 4                  # DataLoader workers
LOG_EVERY = 1_000                # Log every N steps
SAVE_EVERY = 10_000              # Checkpoint every N steps
C_THRESHOLD = 1                  # Min common tags for positive pair

# ── Paths ──
DATA_DIR = DRIVE_BASE
FMA_AUDIO_DIR = DATA_DIR / 'fma_medium'
FMA_META_DIR = DATA_DIR / 'fma_metadata'
MTAT_DIR = DATA_DIR / 'magnatagatune'
MTAT_AUDIO_DIR = MTAT_DIR / 'audio'
GTZAN_DIR = DATA_DIR / 'gtzan'

print(f'\n{"═"*55}')
print(f'Segment: {DURATION}s = {N_SAMPLES:,} samples @ {SR} Hz')
print(f'Batch: {BATCH_SIZE} (sup={int(BATCH_SIZE*BS_RATIO)} + unsup={int(BATCH_SIZE*(1-BS_RATIO))})')
print(f'Training: {MAX_STEPS:,} steps')
print(f'Datos → {DATA_DIR}')
print(f'Resultados → {RESULTS_DIR}')
print(f'{"═"*55}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch 2.10.0+cu128
GPU: NVIDIA A100-SXM4-80GB
VRAM: 79.3 GB

═══════════════════════════════════════════════════════
Segment: 2.7s = 59,535 samples @ 22050 Hz
Batch: 96 (sup=48 + unsup=48)
Training: 200,000 steps
Datos → /content/drive/MyDrive/my_paper_data
Resultados → /content/drive/MyDrive/my_paper_data/results
═══════════════════════════════════════════════════════


## 2. Descarga de Datos

| Dataset | Rol | Tamaño | Fuente |
|---|---|---|---|
| FMA Medium | $\mathcal{U}$ (sin labels) | ~22 GB, 25K clips | os.unil.cloud.switch.ch |
| MagnaTagATune | $\mathcal{S}$ (top 50 tags) | ~600 MB, 25.8K clips | mirg.city.ac.uk |
| GTZAN | Eval. downstream | ~1.2 GB, 1K clips | 🤗 Hugging Face (`marsyas/gtzan`) |

Los datos se descargan a Google Drive y persisten entre sesiones.

In [ ]:
def download_file(url, dest_path, desc=None):
    dest_path = Path(dest_path)
    if dest_path.exists():
        print(f'  ✓ Ya existe: {dest_path.name}')
        return
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    print(f'  ↓ Descargando {desc or dest_path.name}...')
    def _p(bn, bs, ts):
        if ts > 0:
            print(f'\r    {min(100,bn*bs*100/ts):5.1f}% ({bn*bs/1024**3:.2f}/{ts/1024**3:.2f} GB)', end='', flush=True)
    urllib.request.urlretrieve(url, str(dest_path), _p)
    print(f'\n  ✓ Listo: {dest_path.name}')

# ═══════════════════════════════════════
# 2a. FMA Medium (U — self-supervised)
# ═══════════════════════════════════════
print('=== FMA Metadata ===')
meta_zip = DATA_DIR / 'fma_metadata.zip'
download_file('https://os.unil.cloud.switch.ch/fma/fma_metadata.zip', meta_zip, 'FMA Metadata (342 MB)')
if not FMA_META_DIR.exists():
    print('  📦 Extrayendo metadata...')
    with zipfile.ZipFile(str(meta_zip)) as zf:
        zf.extractall(str(DATA_DIR))
    print('  ✓ Extraído')
else:
    print('  ✓ Metadata ya extraída')

print('\n=== FMA Medium Audio ===')
fma_zip = DATA_DIR / 'fma_medium.zip'
download_file('https://os.unil.cloud.switch.ch/fma/fma_medium.zip', fma_zip, 'FMA Medium (~22 GB)')
if not FMA_AUDIO_DIR.exists():
    print('  📦 Extrayendo audio FMA Medium (puede tardar ~30 min)...')
    with zipfile.ZipFile(str(fma_zip)) as zf:
        zf.extractall(str(DATA_DIR))
    print('  ✓ Extraído')
else:
    print('  ✓ Audio FMA ya extraído')

# ═══════════════════════════════════════
# 2b. MagnaTagATune (S — top 50 tags)
# ═══════════════════════════════════════
print('\n=== MagnaTagATune ===')
MTAT_DIR.mkdir(exist_ok=True)
MTAT_URL = 'https://mirg.city.ac.uk/datasets/magnatagatune'
download_file(f'{MTAT_URL}/annotations_final.csv', MTAT_DIR / 'annotations_final.csv', 'annotations')
download_file(f'{MTAT_URL}/clip_info_final.csv', MTAT_DIR / 'clip_info_final.csv', 'clip_info')
for i in range(1, 4):
    download_file(f'{MTAT_URL}/mp3.zip.{i:03d}', MTAT_DIR / f'mp3.zip.{i:03d}', f'mp3.zip.{i:03d} (~200 MB)')

if not MTAT_AUDIO_DIR.exists() or len(list(MTAT_AUDIO_DIR.rglob('*.mp3'))) < 100:
    combined = MTAT_DIR / 'mp3_all.zip'
    if not combined.exists():
        print('  📦 Combinando partes zip...')
        with open(str(combined), 'wb') as outf:
            for i in range(1, 4):
                with open(str(MTAT_DIR / f'mp3.zip.{i:03d}'), 'rb') as inf:
                    outf.write(inf.read())
    print('  📦 Extrayendo audio MTAT...')
    MTAT_AUDIO_DIR.mkdir(exist_ok=True)
    subprocess.run(['unzip', '-qo', str(combined), '-d', str(MTAT_AUDIO_DIR)], check=True)
    print('  ✓ Audio MTAT extraído')
else:
    print('  ✓ Audio MTAT ya extraído')

# ═══════════════════════════════════════
# 2c. GTZAN (evaluación downstream) — via Hugging Face 🤗
# ═══════════════════════════════════════
print('\n=== GTZAN (Hugging Face: marsyas/gtzan) ===')
GTZAN_DIR.mkdir(exist_ok=True)
gtzan_genres_dir = GTZAN_DIR / 'genres'

if not gtzan_genres_dir.exists() or len(list(gtzan_genres_dir.rglob('*.wav'))) < 900:
    import soundfile as sf
    from datasets import load_dataset

    print('  ↓ Descargando GTZAN desde Hugging Face...')
    ds = load_dataset('marsyas/gtzan', 'all', split='train', trust_remote_code=True)
    print(f'  ✓ Descargado: {len(ds)} clips')

    print('  📦 Exportando WAVs a Drive...')
    for item in tqdm(ds, desc='  Exportando GTZAN', leave=False):
        genre = item['genre']
        genre_dir = gtzan_genres_dir / genre
        genre_dir.mkdir(parents=True, exist_ok=True)
        # Use the original filename from the dataset
        filename = Path(item['audio']['path']).name
        out_path = genre_dir / filename
        if not out_path.exists():
            sf.write(str(out_path), item['audio']['array'], item['audio']['sampling_rate'])
    del ds
    print('  ✓ GTZAN exportado a Drive')
else:
    print('  ✓ GTZAN ya extraído')

# ── Validar ──
fma_count = len(list(FMA_AUDIO_DIR.rglob('*.mp3'))) if FMA_AUDIO_DIR.exists() else 0
mtat_count = len(list(MTAT_AUDIO_DIR.rglob('*.mp3'))) if MTAT_AUDIO_DIR.exists() else 0
gtzan_count = len(list(gtzan_genres_dir.rglob('*.wav'))) if gtzan_genres_dir.exists() else 0
print(f'\n{"═"*40}')
print(f'FMA Medium:     {fma_count:>6,} MP3s')
print(f'MagnaTagATune:  {mtat_count:>6,} MP3s')
print(f'GTZAN:          {gtzan_count:>6,} WAVs')
print(f'{"═"*40}')

=== FMA Metadata ===
  ✓ Ya existe: fma_metadata.zip
  ✓ Metadata ya extraída

=== FMA Medium Audio ===
  ✓ Ya existe: fma_medium.zip
  ✓ Audio FMA ya extraído

=== MagnaTagATune ===
  ✓ Ya existe: annotations_final.csv
  ✓ Ya existe: clip_info_final.csv
  ✓ Ya existe: mp3.zip.001
  ✓ Ya existe: mp3.zip.002
  ✓ Ya existe: mp3.zip.003
  ✓ Audio MTAT ya extraído

=== GTZAN ===
  ↓ Descargando GTZAN (~1.2 GB)...


URLError: <urlopen error [Errno 110] Connection timed out>

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3a. Preparar MagnaTagATune (S) — Top 50 tags, split 12:1:3
# ═══════════════════════════════════════════════════════════════
annotations = pd.read_csv(MTAT_DIR / 'annotations_final.csv', sep='\t')
tag_columns = [c for c in annotations.columns if c != 'mp3_path']
print(f'MTAT: {len(annotations):,} clips, {len(tag_columns)} tags')

# ── Top 50 tags (standard MIR benchmark) ──
tag_sums = annotations[tag_columns].sum().sort_values(ascending=False)
TOP50_TAGS = tag_sums.head(50).index.tolist()
print(f'Top 50 tags seleccionados (más frecuente: "{TOP50_TAGS[0]}" con {tag_sums[TOP50_TAGS[0]]:,} clips)')

# ── Canonical 12:1:3 split (folders 0-b train, c val, d-f test) ──
def get_folder(mp3_path):
    parts = str(mp3_path).split('/')
    return parts[0] if len(parts) > 1 else '?'

annotations['folder'] = annotations['mp3_path'].apply(get_folder)
train_folders = list('0123456789ab')
val_folders = ['c']
test_folders = ['d', 'e', 'f']

mtat_train = annotations[annotations['folder'].isin(train_folders)].copy()
mtat_val = annotations[annotations['folder'].isin(val_folders)].copy()
mtat_test = annotations[annotations['folder'].isin(test_folders)].copy()

# ── Resolve audio paths ──
def find_mtat_audio(mp3_path_str):
    p1 = MTAT_AUDIO_DIR / mp3_path_str
    if p1.exists(): return str(p1)
    p2 = MTAT_AUDIO_DIR / 'mp3' / mp3_path_str
    if p2.exists(): return str(p2)
    return None

for df in [mtat_train, mtat_val, mtat_test]:
    df['audio_path'] = df['mp3_path'].apply(find_mtat_audio)
    before = len(df)
    df.dropna(subset=['audio_path'], inplace=True)

print(f'\n=== MTAT Split (con audio válido) ===')
print(f'Train: {len(mtat_train):>6,} clips (folders 0-b)')
print(f'Val:   {len(mtat_val):>6,} clips (folder c)')
print(f'Test:  {len(mtat_test):>6,} clips (folders d-f)')
print(f'Total: {len(mtat_train)+len(mtat_val)+len(mtat_test):>6,}')

# Tags per clip stats
tags_per_clip = mtat_train[TOP50_TAGS].sum(axis=1)
print(f'\nTags/clip (train): media={tags_per_clip.mean():.1f}, mediana={tags_per_clip.median():.0f}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3b. Preparar FMA Medium (U) — sin labels, solo audio
# ═══════════════════════════════════════════════════════════════
tracks_csv = FMA_META_DIR / 'tracks.csv'
tracks_all = pd.read_csv(str(tracks_csv), index_col=0, header=[0, 1])

# Filter to medium subset
fma_medium = tracks_all[tracks_all[('set', 'subset')] == 'medium'].copy()
print(f'FMA Medium metadata: {len(fma_medium):,} tracks')

# Verify audio files exist
def get_fma_audio_path(track_id):
    tid = str(track_id).zfill(6)
    return FMA_AUDIO_DIR / tid[:3] / f'{tid}.mp3'

fma_valid_ids = []
for tid in tqdm(fma_medium.index, desc='Verificando audio FMA', leave=False):
    if get_fma_audio_path(tid).exists():
        fma_valid_ids.append(tid)

fma_df = pd.DataFrame({
    'track_id': fma_valid_ids,
    'audio_path': [str(get_fma_audio_path(tid)) for tid in fma_valid_ids]
})

print(f'FMA Medium con audio válido: {len(fma_df):,} tracks')

# ═══════════════════════════════════════════════════════════════
# 3c. Preparar GTZAN (evaluación downstream)
# ═══════════════════════════════════════════════════════════════
gtzan_genres_dir = GTZAN_DIR / 'genres'
gtzan_data = []
if gtzan_genres_dir.exists():
    gtzan_genre_names = sorted([d.name for d in gtzan_genres_dir.iterdir() if d.is_dir()])
    gtzan_genre_to_idx = {g: i for i, g in enumerate(gtzan_genre_names)}
    for genre_dir in sorted(gtzan_genres_dir.iterdir()):
        if not genre_dir.is_dir():
            continue
        for wav_file in sorted(genre_dir.glob('*.wav')):
            gtzan_data.append({
                'audio_path': str(wav_file),
                'genre': genre_dir.name,
                'genre_idx': gtzan_genre_to_idx[genre_dir.name]
            })
    gtzan_df = pd.DataFrame(gtzan_data)
    print(f'GTZAN: {len(gtzan_df)} clips, {len(gtzan_genre_names)} géneros')
    print(f'  Géneros: {", ".join(gtzan_genre_names)}')
else:
    gtzan_df = pd.DataFrame()
    gtzan_genre_names = []
    print('⚠ GTZAN no disponible')

print(f'\n{"═"*55}')
print(f'RESUMEN DATASETS')
print(f'  U (FMA Medium):     {len(fma_df):>6,} tracks — sin labels')
print(f'  S (MTAT train):     {len(mtat_train):>6,} clips — {len(TOP50_TAGS)} tags')
print(f'  S (MTAT val):       {len(mtat_val):>6,} clips')
print(f'  S (MTAT test):      {len(mtat_test):>6,} clips')
print(f'  Eval (GTZAN):       {len(gtzan_df):>6,} clips — {len(gtzan_genre_names)} géneros')
print(f'{"═"*55}')

## 4. Dataset & DataLoader

Cada sample produce **2 segmentos adyacentes no-solapados de 2.7s** como par positivo (paper Section 4.2).  
- Samples de MTAT ($\mathcal{S}$): `is_supervised=True`, tag vector de 50 dims  
- Samples de FMA ($\mathcal{U}$): `is_supervised=False`, tag vector = ceros

In [ ]:
class AudioDataset(Dataset):
    """
    Base audio dataset. Loads audio, returns 2 adjacent non-overlapping
    segments of DURATION seconds as a positive pair.
    """
    def __init__(self, audio_paths, tag_vectors=None, is_supervised=True,
                 sr=SR, duration=DURATION):
        self.audio_paths = list(audio_paths)
        self.tag_vectors = tag_vectors  # (N, 50) numpy array or None
        self.is_supervised = is_supervised
        self.sr = sr
        self.n_samples = int(sr * duration)

    def __len__(self):
        return len(self.audio_paths)

    def _load_audio(self, path):
        try:
            waveform, orig_sr = torchaudio.load(path)
        except Exception:
            y, _ = librosa.load(path, sr=self.sr, mono=True)
            return torch.from_numpy(y).float()
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if orig_sr != self.sr:
            waveform = torchaudio.transforms.Resample(orig_sr, self.sr)(waveform)
        return waveform.squeeze(0)

    def _get_adjacent_pair(self, waveform):
        """Extract 2 adjacent non-overlapping segments (paper Section 4.2)."""
        total_needed = 2 * self.n_samples
        if len(waveform) < total_needed:
            waveform = F.pad(waveform, (0, total_needed - len(waveform)))
        max_start = len(waveform) - total_needed
        start = random.randint(0, max(0, max_start))
        seg1 = waveform[start : start + self.n_samples]
        seg2 = waveform[start + self.n_samples : start + 2 * self.n_samples]
        return seg1, seg2

    def __getitem__(self, idx):
        waveform = self._load_audio(self.audio_paths[idx])
        seg1, seg2 = self._get_adjacent_pair(waveform)

        if self.tag_vectors is not None:
            tags = torch.from_numpy(self.tag_vectors[idx]).float()
        else:
            tags = torch.zeros(len(TOP50_TAGS), dtype=torch.float32)

        return seg1, seg2, tags, self.is_supervised


class SemiSupConBatchSampler(Sampler):
    """
    Yields batches with controlled proportion bs_ratio of supervised samples.
    Cycles through both sets independently, reshuffling when exhausted.
    """
    def __init__(self, n_supervised, n_unsupervised, batch_size, bs_ratio, max_batches=None):
        self.n_sup = n_supervised
        self.n_unsup = n_unsupervised
        self.batch_size = batch_size
        self.n_sup_per_batch = max(1, int(batch_size * bs_ratio))
        self.n_unsup_per_batch = batch_size - self.n_sup_per_batch
        # max_batches limits iteration (for step-based training)
        self.max_batches = max_batches

    def __iter__(self):
        sup_indices = list(range(self.n_sup))
        unsup_indices = list(range(self.n_sup, self.n_sup + self.n_unsup))
        random.shuffle(sup_indices)
        random.shuffle(unsup_indices)
        si, ui = 0, 0
        count = 0

        while True:
            if self.max_batches and count >= self.max_batches:
                return
            batch = []
            for _ in range(self.n_sup_per_batch):
                if si >= len(sup_indices):
                    random.shuffle(sup_indices)
                    si = 0
                batch.append(sup_indices[si]); si += 1
            for _ in range(self.n_unsup_per_batch):
                if ui >= len(unsup_indices):
                    random.shuffle(unsup_indices)
                    ui = 0
                batch.append(unsup_indices[ui]); ui += 1
            yield batch
            count += 1

    def __len__(self):
        if self.max_batches:
            return self.max_batches
        return max(self.n_sup // self.n_sup_per_batch,
                   self.n_unsup // self.n_unsup_per_batch)


# ── Build datasets ──
# MTAT supervised dataset (train split only for pretraining)
mtat_tag_vectors = mtat_train[TOP50_TAGS].values.astype(np.float32)
mtat_audio_paths = mtat_train['audio_path'].tolist()
mtat_dataset = AudioDataset(mtat_audio_paths, mtat_tag_vectors, is_supervised=True)

# FMA unsupervised dataset
fma_audio_paths = fma_df['audio_path'].tolist()
fma_dataset = AudioDataset(fma_audio_paths, tag_vectors=None, is_supervised=False)

# Combined dataset: [MTAT (0..n_sup-1), FMA (n_sup..n_sup+n_unsup-1)]
combined_dataset = ConcatDataset([mtat_dataset, fma_dataset])

print(f'MTAT dataset (S): {len(mtat_dataset):,} clips')
print(f'FMA  dataset (U): {len(fma_dataset):,} clips')
print(f'Combined:         {len(combined_dataset):,} clips')

# Test loading
seg1, seg2, tags, is_sup = combined_dataset[0]
print(f'\nSample supervised — seg1: {seg1.shape}, seg2: {seg2.shape}, tags sum: {tags.sum():.0f}, supervised: {is_sup}')
seg1, seg2, tags, is_sup = combined_dataset[len(mtat_dataset)]
print(f'Sample unsupervised — seg1: {seg1.shape}, tags sum: {tags.sum():.0f}, supervised: {is_sup}')

## 5. Augmentation Pipeline (Table 1 del paper)

Gain → Polarity Inv. → Colored Noise → Random Filter (one-of) → Pitch Shift → Delay

In [ ]:
class AudioAugmentations:
    """Stochastic augmentation chain — Table 1 of SemiSupCon paper."""

    def __init__(self, sr=SR):
        self.sr = sr

    def gain(self, x, p=0.4):
        if random.random() > p: return x
        return x * 10 ** (random.uniform(-15, 5) / 20)

    def polarity_inversion(self, x, p=0.6):
        if random.random() > p: return x
        return -x

    def colored_noise(self, x, p=0.6):
        if random.random() > p: return x
        snr_db = random.uniform(3, 30)
        noise = torch.randn_like(x)
        kind = random.choice(['white', 'pink', 'brown'])
        if kind == 'pink':
            fft = torch.fft.rfft(noise)
            freqs = torch.arange(1, len(fft) + 1, dtype=torch.float32)
            noise = torch.fft.irfft(fft / freqs.sqrt(), n=len(x))
        elif kind == 'brown':
            noise = torch.cumsum(noise, 0)
            noise = noise / (noise.abs().max() + 1e-8)
        sig_pow = (x ** 2).mean()
        noi_pow = (noise ** 2).mean()
        if noi_pow > 0:
            noise = noise * torch.sqrt(sig_pow / (10 ** (snr_db / 10) * noi_pow))
        return x + noise

    def random_filter(self, x, p=0.3):
        if random.random() > p: return x
        nyq = self.sr / 2
        try:
            ftype = random.choice(['low', 'high', 'band', 'bandstop'])
            if ftype == 'low':
                b, a = butter(2, min(random.uniform(1000, 8000) / nyq, 0.99), btype='low')
            elif ftype == 'high':
                b, a = butter(2, max(random.uniform(100, 2000) / nyq, 0.01), btype='high')
            else:
                lo = random.uniform(200, 2000)
                hi = random.uniform(lo + 500, min(lo + 5000, nyq - 1))
                b, a = butter(2, [lo / nyq, hi / nyq], btype=ftype)
            out = lfilter(b, a, x.numpy())
            return torch.from_numpy(out.copy()).float()
        except Exception:
            return x

    def pitch_shift(self, x, p=0.6):
        if random.random() > p: return x
        n_steps = random.uniform(-4, 4)
        shifted = librosa.effects.pitch_shift(y=x.numpy(), sr=self.sr, n_steps=n_steps)
        return torch.from_numpy(shifted).float()

    def delay(self, x, p=0.6):
        """Simple delay/echo effect — Table 1."""
        if random.random() > p: return x
        delay_ms = random.uniform(100, 500)
        delay_samples = int(self.sr * delay_ms / 1000)
        n_reflections = random.randint(1, 3)
        atten_db = random.uniform(-6, -3)
        wet_dry = random.uniform(0.25, 1.0)
        out = x.clone()
        for r in range(1, n_reflections + 1):
            offset = delay_samples * r
            if offset >= len(x): break
            gain = 10 ** (atten_db * r / 20) * wet_dry
            out[offset:] += gain * x[:len(x) - offset]
        # Normalize to prevent clipping
        peak = out.abs().max()
        if peak > 1.0:
            out = out / peak
        return out

    def __call__(self, x):
        x = self.gain(x)
        x = self.polarity_inversion(x)
        x = self.colored_noise(x)
        x = self.random_filter(x)
        x = self.pitch_shift(x)
        x = self.delay(x)
        return x

augmenter = AudioAugmentations(sr=SR)

# Quick test
test_seg = combined_dataset[0][0]
aug_seg = augmenter(test_seg)
fig, axes = plt.subplots(2, 1, figsize=(12, 3))
axes[0].plot(test_seg.numpy()[:2000], linewidth=0.4); axes[0].set_title('Original'); axes[0].set_ylabel('Amp')
axes[1].plot(aug_seg.numpy()[:2000], linewidth=0.4, color='orange'); axes[1].set_title('Augmented'); axes[1].set_ylabel('Amp')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'augmentation_example.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Augmentation pipeline ✓')

## 5. SampleCNN Encoder

In [ ]:
class SampleCNNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, pool=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, k, stride=1, padding=k // 2),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(pool)
        )
    def forward(self, x):
        return self.net(x)


class SampleCNN(nn.Module):
    """
    SampleCNN encoder (Lee et al., 2018).
    First strided conv + 9 conv blocks with MaxPool(3) + GAP → d_E=512.
    """
    def __init__(self, d_encoder=D_ENCODER):
        super().__init__()
        self.first_conv = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=3, stride=3, padding=0),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True)
        )
        channels = [128, 128, 128, 256, 256, 256, 256, 256, 512]
        self.blocks = nn.ModuleList()
        in_ch = 128
        for out_ch in channels:
            self.blocks.append(SampleCNNBlock(in_ch, out_ch))
            in_ch = out_ch
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.d_encoder = d_encoder

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = self.first_conv(x)
        for block in self.blocks:
            x = block(x)
        return self.gap(x).squeeze(-1)


# Test
encoder = SampleCNN().to(DEVICE)
test_in = torch.randn(4, N_SAMPLES, device=DEVICE)
test_out = encoder(test_in)
print(f'SampleCNN: {test_in.shape} → {test_out.shape}')
print(f'Parameters: {sum(p.numel() for p in encoder.parameters()):,}')
del encoder, test_in, test_out
torch.cuda.empty_cache()

## 6. Projection Head

In [ ]:
class ProjectionHead(nn.Module):
    """2-layer MLP: d_E → d_E → d_g + L2 normalization."""
    def __init__(self, d_in=D_ENCODER, d_out=D_PROJ):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_in, d_in), nn.ReLU(inplace=True),
            nn.Linear(d_in, d_out)
        )
    def forward(self, x):
        return F.normalize(self.mlp(x), dim=-1)


class SemiSupConModel(nn.Module):
    """Encoder + Projection Head."""
    def __init__(self):
        super().__init__()
        self.encoder = SampleCNN()
        self.projector = ProjectionHead()

    def forward(self, x):
        h = self.encoder(x)
        z = self.projector(h)
        return h, z

model = SemiSupConModel().to(DEVICE)
test_in = torch.randn(4, N_SAMPLES, device=DEVICE)
h, z = model(test_in)
print(f'Encoder h: {h.shape}, Projection z: {z.shape}')
print(f'z L2-normalized: {torch.allclose(z.norm(dim=-1), torch.ones(4, device=DEVICE), atol=1e-5)}')
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')
del test_in, h, z
torch.cuda.empty_cache()

## 7. SemiSupCon Loss (Equation 3)

$$\mathcal{L}_{sem}^i = \frac{-1}{|P_A(i)|} \sum_{p \in P_A(i)} \log \frac{\sigma_{i,p}}{\sum_{n \in N(i) \cup P_A(i)} \sigma_{i,n}}$$

- **Supervised** ($i \in \mathcal{S}$): $P_A(i) = P_s(i) \cup P_u(i)$ — positivos por tags ($C \geq 1$) **+** par de augmentation
- **Unsupervised** ($i \in \mathcal{U}$): $P_A(i) = P_u(i) = \{p(i)\}$ — solo par de augmentation
- Positive mask **vectorizada** para eficiencia en GPU: `tags @ tags.T >= C`

In [ ]:
class SemiSupConLoss(nn.Module):
    """
    Semi-Supervised Contrastive Loss — multi-label, fully vectorized.

    For a batch of N samples with 2 views → 2N representations.
    Positive mask:
      - Augmentation pairs (i, i+N) always positive
      - For supervised pairs: positive if they share >= C common tags
    """
    def __init__(self, temperature=TAU, C=C_THRESHOLD):
        super().__init__()
        self.temperature = temperature
        self.C = C

    def forward(self, z1, z2, tag_vectors, is_supervised):
        """
        Args:
            z1, z2: (N, d) L2-normalized projections
            tag_vectors: (N, n_tags) binary tag vectors
            is_supervised: (N,) bool tensor
        Returns:
            scalar loss
        """
        N = z1.size(0)
        device = z1.device

        # ── Concatenate views: [z1_0..z1_N, z2_0..z2_N] → (2N, d) ──
        z = torch.cat([z1, z2], dim=0)

        # ── Pairwise similarities / τ ──
        sim = z @ z.T / self.temperature  # (2N, 2N)

        # ── Self-mask: exclude diagonal ──
        self_mask = ~torch.eye(2 * N, dtype=torch.bool, device=device)

        # ── Augmentation positive mask ──
        # (i, i+N) and (i+N, i) are always positive
        aug_mask = torch.zeros(2 * N, 2 * N, dtype=torch.bool, device=device)
        idx = torch.arange(N, device=device)
        aug_mask[idx, idx + N] = True
        aug_mask[idx + N, idx] = True

        # ── Supervised positive mask (vectorized) ──
        # Two supervised samples are positive if they share >= C tags
        sup_mask = torch.zeros(2 * N, 2 * N, dtype=torch.bool, device=device)
        tags_ext = torch.cat([tag_vectors, tag_vectors], dim=0)  # (2N, n_tags)
        is_sup_ext = torch.cat([is_supervised, is_supervised], dim=0)  # (2N,)

        if is_sup_ext.any():
            # Compute common tags matrix for all pairs
            common_tags = tags_ext @ tags_ext.T  # (2N, 2N) — count of shared tags
            # Both must be supervised and share >= C tags
            both_sup = is_sup_ext.unsqueeze(1) & is_sup_ext.unsqueeze(0)  # (2N, 2N)
            sup_mask = both_sup & (common_tags >= self.C)

        # ── Combined positive mask ──
        pos_mask = (aug_mask | sup_mask) & self_mask  # exclude self

        # ── Numerical stability ──
        logits_max, _ = sim.max(dim=1, keepdim=True)
        logits = sim - logits_max.detach()

        # ── Log-prob: log(exp(sim_ip) / sum_k exp(sim_ik)) for k != i ──
        exp_logits = torch.exp(logits) * self_mask.float()
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-12)

        # ── Mean over positives, then mean over batch ──
        pos_count = pos_mask.sum(dim=1).float()
        valid = pos_count > 0
        mean_log_prob = (pos_mask.float() * log_prob).sum(dim=1) / (pos_count + 1e-12)

        return -mean_log_prob[valid].mean()


# ── Test ──
criterion = SemiSupConLoss(temperature=TAU, C=C_THRESHOLD)
fake_z1 = F.normalize(torch.randn(BATCH_SIZE, D_PROJ, device=DEVICE), dim=-1)
fake_z2 = F.normalize(torch.randn(BATCH_SIZE, D_PROJ, device=DEVICE), dim=-1)
n_sup = BATCH_SIZE // 2
fake_tags = torch.zeros(BATCH_SIZE, 50, device=DEVICE)
fake_tags[:n_sup, :5] = (torch.rand(n_sup, 5, device=DEVICE) > 0.5).float()
fake_is_sup = torch.zeros(BATCH_SIZE, dtype=torch.bool, device=DEVICE)
fake_is_sup[:n_sup] = True

test_loss = criterion(fake_z1, fake_z2, fake_tags, fake_is_sup)
print(f'Test loss (batch={BATCH_SIZE}): {test_loss.item():.4f}')
print(f'Finite: {torch.isfinite(test_loss).item()} ✓')
del fake_z1, fake_z2, fake_tags, fake_is_sup
torch.cuda.empty_cache()

## 8. Training Loop

- **200K steps** (paper Section 4.3), no epochs
- Mixed precision (AMP) para velocidad en A100/V100
- Checkpoint cada 10K steps → Google Drive
- Log cada 1K steps

In [ ]:
# ── Check for existing checkpoint to resume ──
resume_step = 0
resume_ckpt = CKPT_DIR / 'latest_semisupcon.pt'
if resume_ckpt.exists():
    ckpt = torch.load(str(resume_ckpt), map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    resume_step = ckpt['step']
    print(f'✓ Resuming from step {resume_step:,}')
    del ckpt
    torch.cuda.empty_cache()

# ── Optimizer & scaler ──
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler('cuda')

# ── DataLoader ──
batch_sampler = SemiSupConBatchSampler(
    n_supervised=len(mtat_dataset),
    n_unsupervised=len(fma_dataset),
    batch_size=BATCH_SIZE,
    bs_ratio=BS_RATIO,
    max_batches=MAX_STEPS - resume_step
)

def collate_fn(batch):
    seg1s, seg2s, tags, is_sups = zip(*batch)
    return (torch.stack(seg1s), torch.stack(seg2s),
            torch.stack(tags), torch.tensor(is_sups, dtype=torch.bool))

train_loader = DataLoader(
    combined_dataset,
    batch_sampler=batch_sampler,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')
print(f'Steps remaining: {MAX_STEPS - resume_step:,}')
print(f'Batch: {BATCH_SIZE} (sup={int(BATCH_SIZE*BS_RATIO)} + unsup={int(BATCH_SIZE*(1-BS_RATIO))})')

# ═══════════════════════════════════════════
# Training loop — step-based, 200K steps
# ═══════════════════════════════════════════
model.train()
loss_history = []
best_loss = float('inf')
step = resume_step
t0 = time.time()

pbar = tqdm(train_loader, total=MAX_STEPS - resume_step, desc='Training', initial=0)

for seg1_batch, seg2_batch, tags_batch, is_sup_batch in pbar:
    step += 1

    # Augment both views independently (on CPU, then move to GPU)
    view1 = torch.stack([augmenter(s) for s in seg1_batch]).to(DEVICE)
    view2 = torch.stack([augmenter(s) for s in seg2_batch]).to(DEVICE)
    tags_batch = tags_batch.to(DEVICE)
    is_sup_batch = is_sup_batch.to(DEVICE)

    # Forward + loss with mixed precision
    with torch.amp.autocast('cuda'):
        _, z1 = model(view1)
        _, z2 = model(view2)
        loss = criterion(z1, z2, tags_batch, is_sup_batch)

    # Backward
    optimizer.zero_grad()
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    loss_val = loss.item()
    loss_history.append(loss_val)

    # ── Logging ──
    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        steps_sec = step / elapsed if elapsed > 0 else 0
        eta_h = (MAX_STEPS - step) / steps_sec / 3600 if steps_sec > 0 else 0
        avg_loss = np.mean(loss_history[-LOG_EVERY:])
        pbar.set_postfix({
            'loss': f'{avg_loss:.4f}',
            's/s': f'{steps_sec:.1f}',
            'ETA': f'{eta_h:.1f}h'
        })

    # ── Checkpoint ──
    if step % SAVE_EVERY == 0:
        ckpt_data = {
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss_history': loss_history,
            'best_loss': best_loss,
        }
        torch.save(ckpt_data, str(CKPT_DIR / f'semisupcon_step{step}.pt'))
        torch.save(ckpt_data, str(resume_ckpt))
        # Save loss history as JSON
        with open(str(RESULTS_DIR / 'loss_history.json'), 'w') as f:
            json.dump(loss_history, f)
        print(f'\n  💾 Checkpoint saved: step {step:,}, avg_loss={np.mean(loss_history[-SAVE_EVERY:]):.4f}')

    # ── Best model ──
    if step >= 1000 and step % 1000 == 0:
        recent_avg = np.mean(loss_history[-1000:])
        if recent_avg < best_loss:
            best_loss = recent_avg
            torch.save({
                'step': step,
                'model_state_dict': model.state_dict(),
                'loss': best_loss,
            }, str(CKPT_DIR / 'best_semisupcon.pt'))

pbar.close()
elapsed_total = time.time() - t0
print(f'\n✓ Training complete: {MAX_STEPS:,} steps in {elapsed_total/3600:.1f}h')
print(f'  Best loss: {best_loss:.4f}')
print(f'  Final avg loss (last 1K): {np.mean(loss_history[-1000:]):.4f}')

# Save final checkpoint
torch.save({
    'step': step,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss_history': loss_history,
    'best_loss': best_loss,
}, str(CKPT_DIR / 'final_semisupcon.pt'))
with open(str(RESULTS_DIR / 'loss_history.json'), 'w') as f:
    json.dump(loss_history, f)
print(f'  💾 Final checkpoint saved to Drive')

## 9. Training Loss Visualization

In [ ]:
# Load loss history (in case we're in a new session)
loss_json = RESULTS_DIR / 'loss_history.json'
if 'loss_history' not in dir() or len(loss_history) == 0:
    if loss_json.exists():
        with open(str(loss_json)) as f:
            loss_history = json.load(f)
        print(f'Loaded {len(loss_history):,} loss values from Drive')

def moving_avg(data, w):
    if len(data) < w: return data
    c = np.cumsum(np.insert(data, 0, 0))
    return (c[w:] - c[:-w]) / w

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Full loss curve (subsampled for readability)
stride = max(1, len(loss_history) // 2000)
x_sub = list(range(0, len(loss_history), stride))
y_sub = [loss_history[i] for i in x_sub]
axes[0].plot(x_sub, y_sub, alpha=0.15, color='#4a90d9', linewidth=0.5, label='Raw')
if len(loss_history) >= 5000:
    sm = moving_avg(loss_history, 5000)
    axes[0].plot(range(2500, 2500 + len(sm)), sm, color='#e74c3c', linewidth=2, label='MA(5K)')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('SemiSupCon Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Last 50K steps zoomed
if len(loss_history) > 50_000:
    last_50k = loss_history[-50_000:]
    sm50 = moving_avg(last_50k, 1000)
    axes[1].plot(range(len(loss_history)-50_000, len(loss_history)), last_50k,
                 alpha=0.1, color='#4a90d9', linewidth=0.3)
    axes[1].plot(range(len(loss_history)-50_000+500, len(loss_history)-50_000+500+len(sm50)),
                 sm50, color='#e74c3c', linewidth=2, label='MA(1K)')
    axes[1].set_xlabel('Step')
    axes[1].set_title('Last 50K Steps (zoom)', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
else:
    axes[1].text(0.5, 0.5, f'Only {len(loss_history):,} steps\n(zoom at >50K)',
                 ha='center', va='center', fontsize=14, transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'training_loss.png'), dpi=200, bbox_inches='tight')
plt.show()

print(f'Steps: {len(loss_history):,}')
print(f'Initial loss (first 100): {np.mean(loss_history[:100]):.4f}')
print(f'Final loss (last 1K):     {np.mean(loss_history[-1000:]):.4f}')
print(f'Best loss:                {best_loss:.4f}')
print(f'📊 Saved → {RESULTS_DIR / "training_loss.png"}')

## 10. Evaluation — MTAT Tagging (AUROC, AP)

Linear probing con frozen encoder:
- Extract embeddings para train/val/test
- Train 2-layer MLP (512→256→50, sigmoid, BCE loss)
- Early stopping on val AUROC
- Report AUROC & AP on test set

In [ ]:
# ── Load best checkpoint ──
best_ckpt = CKPT_DIR / 'best_semisupcon.pt'
if best_ckpt.exists():
    ckpt = torch.load(str(best_ckpt), map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"✓ Loaded best model (step {ckpt.get('step', '?')}, loss={ckpt.get('loss', '?'):.4f})")
    del ckpt
else:
    print("⚠ No best checkpoint found, using current model state")

# ── Freeze encoder ──
model.eval()
for p in model.parameters():
    p.requires_grad = False

@torch.no_grad()
def extract_embeddings(df, batch_size=128):
    """Extract encoder embeddings for a DataFrame with 'audio_path' column."""
    embeddings = []
    for i in tqdm(range(0, len(df), batch_size), desc='Extracting embeddings'):
        batch_paths = df['audio_path'].iloc[i:i+batch_size].tolist()
        waveforms = []
        for p in batch_paths:
            wav, sr_orig = torchaudio.load(p)
            wav = wav.mean(0)  # mono
            if sr_orig != SR:
                wav = torchaudio.functional.resample(wav, sr_orig, SR)
            # Take center crop of N_SAMPLES
            if wav.shape[0] >= N_SAMPLES:
                start = (wav.shape[0] - N_SAMPLES) // 2
                wav = wav[start:start + N_SAMPLES]
            else:
                wav = F.pad(wav, (0, N_SAMPLES - wav.shape[0]))
            waveforms.append(wav)
        batch_tensor = torch.stack(waveforms).to(DEVICE)
        h = model.encoder(batch_tensor)
        embeddings.append(h.cpu())
    return torch.cat(embeddings, dim=0)

print('Extracting MTAT embeddings...')
X_train = extract_embeddings(mtat_train)
X_val   = extract_embeddings(mtat_val)
X_test  = extract_embeddings(mtat_test)

Y_train = torch.tensor(mtat_train[TOP50_TAGS].values, dtype=torch.float32)
Y_val   = torch.tensor(mtat_val[TOP50_TAGS].values, dtype=torch.float32)
Y_test  = torch.tensor(mtat_test[TOP50_TAGS].values, dtype=torch.float32)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

In [ ]:
# ── 2-layer MLP probe for multi-label classification ──
class MLPProbe(nn.Module):
    def __init__(self, d_in=D_ENCODER, d_hidden=256, n_classes=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(d_hidden, n_classes)
        )
    def forward(self, x):
        return self.net(x)

probe = MLPProbe().to(DEVICE)
probe_optimizer = torch.optim.Adam(probe.parameters(), lr=3e-4)
probe_criterion = nn.BCEWithLogitsLoss()

# ── Training with early stopping ──
train_ds = TensorDataset(X_train, Y_train)
val_ds   = TensorDataset(X_val, Y_val)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=256)

best_val_auroc = 0
patience = 10
wait = 0
probe_history = {'train_loss': [], 'val_auroc': [], 'val_ap': []}

for epoch in range(200):
    # Train
    probe.train()
    epoch_loss = []
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = probe(xb)
        loss = probe_criterion(logits, yb)
        probe_optimizer.zero_grad()
        loss.backward()
        probe_optimizer.step()
        epoch_loss.append(loss.item())
    probe_history['train_loss'].append(np.mean(epoch_loss))

    # Validate
    probe.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_dl:
            logits = probe(xb.to(DEVICE))
            all_preds.append(torch.sigmoid(logits).cpu())
            all_labels.append(yb)
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    val_auroc = roc_auc_score(all_labels, all_preds, average='macro')
    val_ap = average_precision_score(all_labels, all_preds, average='macro')
    probe_history['val_auroc'].append(val_auroc)
    probe_history['val_ap'].append(val_ap)

    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        wait = 0
        torch.save(probe.state_dict(), str(CKPT_DIR / 'best_mtat_probe.pt'))
    else:
        wait += 1

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1}: loss={np.mean(epoch_loss):.4f}, AUROC={val_auroc:.4f}, AP={val_ap:.4f}')

    if wait >= patience:
        print(f'  Early stopping at epoch {epoch+1}')
        break

print(f'\nBest val AUROC: {best_val_auroc:.4f}')

In [ ]:
# ── Test set evaluation ──
probe.load_state_dict(torch.load(str(CKPT_DIR / 'best_mtat_probe.pt'), map_location=DEVICE, weights_only=True))
probe.eval()

test_dl = DataLoader(TensorDataset(X_test, Y_test), batch_size=256)
all_preds, all_labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        logits = probe(xb.to(DEVICE))
        all_preds.append(torch.sigmoid(logits).cpu())
        all_labels.append(yb)

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

mtat_auroc = roc_auc_score(all_labels, all_preds, average='macro')
mtat_ap = average_precision_score(all_labels, all_preds, average='macro')

print('╔══════════════════════════════════════════╗')
print('║       MTAT Test Set Results              ║')
print('╠══════════════════════════════════════════╣')
print(f'║  AUROC (macro):  {mtat_auroc:.4f}                 ║')
print(f'║  AP    (macro):  {mtat_ap:.4f}                 ║')
print('╠══════════════════════════════════════════╣')
print(f'║  Paper SSL:      0.8880                 ║')
print(f'║  Paper SemiSup:  0.8970                 ║')
print('╚══════════════════════════════════════════╝')

# Save results
mtat_results = {'auroc': mtat_auroc, 'ap': mtat_ap}
with open(str(RESULTS_DIR / 'mtat_results.json'), 'w') as f:
    json.dump(mtat_results, f, indent=2)
print(f'📊 Saved → {RESULTS_DIR / "mtat_results.json"}')

In [ ]:
# ═══════════════════════════════════════════
# 11. Evaluation — GTZAN Genre Classification
# ═══════════════════════════════════════════
print('Extracting GTZAN embeddings...')
X_gtzan = extract_embeddings(gtzan_df)
Y_gtzan = torch.tensor(gtzan_df['genre_idx'].values, dtype=torch.long)
print(f'GTZAN: {X_gtzan.shape}, {len(gtzan_genre_names)} genres')

# ── Train/test split (80/20 stratified) ──
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(X_gtzan, Y_gtzan))

X_gz_train, X_gz_test = X_gtzan[train_idx], X_gtzan[test_idx]
Y_gz_train, Y_gz_test = Y_gtzan[train_idx], Y_gtzan[test_idx]

# ── MLP probe for classification ──
gtzan_probe = nn.Sequential(
    nn.Linear(D_ENCODER, 256), nn.ReLU(True), nn.Dropout(0.3),
    nn.Linear(256, len(gtzan_genre_names))
).to(DEVICE)

gz_optimizer = torch.optim.Adam(gtzan_probe.parameters(), lr=3e-4)
gz_criterion = nn.CrossEntropyLoss()

gz_train_ds = TensorDataset(X_gz_train, Y_gz_train)
gz_train_dl = DataLoader(gz_train_ds, batch_size=128, shuffle=True)
gz_test_ds  = TensorDataset(X_gz_test, Y_gz_test)
gz_test_dl  = DataLoader(gz_test_ds, batch_size=128)

best_gz_acc = 0
for epoch in range(200):
    gtzan_probe.train()
    for xb, yb in gz_train_dl:
        logits = gtzan_probe(xb.to(DEVICE))
        loss = gz_criterion(logits, yb.to(DEVICE))
        gz_optimizer.zero_grad()
        loss.backward()
        gz_optimizer.step()

    # Eval
    gtzan_probe.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in gz_test_dl:
            preds = gtzan_probe(xb.to(DEVICE)).argmax(1).cpu()
            correct += (preds == yb).sum().item()
            total += len(yb)
    acc = correct / total
    if acc > best_gz_acc:
        best_gz_acc = acc
        torch.save(gtzan_probe.state_dict(), str(CKPT_DIR / 'best_gtzan_probe.pt'))

    if (epoch + 1) % 20 == 0:
        print(f'  Epoch {epoch+1}: acc={acc:.4f}')

print(f'\n╔══════════════════════════════════════════╗')
print(f'║       GTZAN Test Results                 ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Top-1 Accuracy: {best_gz_acc:.4f}                ║')
print(f'╠══════════════════════════════════════════╣')
print(f'║  Paper SSL:      0.7630                 ║')
print(f'║  Paper SemiSup:  0.7550                 ║')
print(f'╚══════════════════════════════════════════╝')

gtzan_results = {'accuracy': best_gz_acc}
with open(str(RESULTS_DIR / 'gtzan_results.json'), 'w') as f:
    json.dump(gtzan_results, f, indent=2)
print(f'📊 Saved → {RESULTS_DIR / "gtzan_results.json"}')

In [ ]:
# ═══════════════════════════════════════════
# 12. t-SNE Visualization of GTZAN Embeddings
# ═══════════════════════════════════════════
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
emb_2d = tsne.fit_transform(X_gtzan.numpy())

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(emb_2d[:, 0], emb_2d[:, 1],
                     c=Y_gtzan.numpy(), cmap='tab10', s=12, alpha=0.7)

# Legend
handles = [plt.Line2D([0], [0], marker='o', color='w',
           markerfacecolor=plt.cm.tab10(i/10), markersize=8,
           label=g) for i, g in enumerate(gtzan_genre_names)]
ax.legend(handles=handles, loc='best', fontsize=8, ncol=2, framealpha=0.8)

ax.set_title('t-SNE of SemiSupCon Encoder Embeddings (GTZAN)', fontweight='bold', fontsize=13)
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'tsne_gtzan.png'), dpi=200, bbox_inches='tight')
plt.show()
print(f'📊 Saved → {RESULTS_DIR / "tsne_gtzan.png"}')

## 13. Results Summary

| Metric | Paper (SSL) | Paper (SemiSup) | **Ours** |
|---|---|---|---|
| **MTAT AUROC** | 88.8 | 89.7 | *see above* |
| **MTAT AP** | — | — | *see above* |
| **GTZAN Accuracy** | 76.3 | 75.5 | *see above* |

All checkpoints, loss curves, and results saved to Google Drive:
- `checkpoints/` — model weights
- `results/` — metrics JSON, loss plots, t-SNE

Reference: Guinot, Quinton & Fazekas, *"SemiSupCon: Semi-Supervised Contrastive Learning for Music"*, ISMIR 2024.